# PRISM Replication Notebook
## Predictive Renal Intelligence Survival Modeling — CKD Stage 5 Dialysis Decision Aid

This notebook provides a step-by-step guide for **academic reviewers and researchers** to:
1. Load the trained PRISM models
2. Reproduce single-patient predictions
3. Verify key performance metrics from the manuscript
4. Confirm model agreement (CF vs RL) and zone assignment logic

### Prerequisites
```bash
pip install -r requirements.txt
./setup_models.sh          # populate models/ from development project
```

### Citation
If you use PRISM in your research, please cite:
> Leung KC et al. PRISM: A Multi-Model Causal Survival Analysis Framework for
> Individualised Dialysis Decision Support in Advanced Chronic Kidney Disease.
> *[Journal], 2025.*

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'prism_deploy'))

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
print('Environment ready.')

## 1. Load Models

All four model groups are loaded once and cached. First call takes ~10–30 seconds.

In [ ]:
from app.predict import load_preprocessor, load_models

print('Loading preprocessor (MICE imputer + MinMaxScaler) ...')
prep = load_preprocessor()
print(f'  Features: {prep["features"]}')
print(f'  Continuous features (scaled): {prep["continuous"]}')

print('\nLoading models (RSF DR-Learner, CF, RL, ACMM) ...')
models = load_models()
print(f'  Loaded: {list(models.keys())}')
print('Models ready.')

## 2. Feature Extraction Rationale

The 6 features were selected via a 3-stage process (§2.4.2):

| Feature | Domain | Extraction window | Missing (%) | Selection reason |
|---------|--------|-------------------|-------------|------------------|
| Age at t₀ | Demographics | Static | 0.0% | Core prognostic, no collinearity |
| Sex | Demographics | Static | 0.0% | CKD-EPI component; prognostic |
| Creatinine (µmol/L) | Renal function | 90-day lookback | 0.0% | eGFR denominator; available at all 41 centres |
| Haemoglobin (g/dL) | Anaemia | 90-day lookback | 3.2% | LASSO-selected; low missingness |
| Phosphate (mmol/L) | Mineral metabolism | 90-day lookback | 25.1% | LASSO-selected; imputed |
| CCI total score | Comorbidity | 5-year lookback | 0.0% | Strong SHAP importance; cross-centre available |

**Excluded despite clinical relevance**: albumin, calcium, bicarbonate, UACR (unavailable at all 41 QMH centres); eGFR (collinear with creatinine+age+sex); HbA1c (VIF=10.67, unreliable in CKD anaemia).

## 3. Single-Patient Inference

We demonstrate three representative patients spanning the four clinical zones.

In [ ]:
from app.predict import predict_patient

# Representative patients (fabricated, not real)
demo_patients = [
    dict(label='Zone B — Young male, DM+CHF, high Cr (expected: low ACMM, strong benefit)',
         age=55, female=0, hb=8.1, po4=2.1, cci=2, cr=720,
         cci_flags={'diabetes_wo_complication': 1, 'congestive_heart_failure': 1}),
    dict(label='Zone C/D — Mid-age female, DM complications + CVD',
         age=68, female=1, hb=9.5, po4=1.7, cci=4, cr=580,
         cci_flags={'diabetes_w_complication': 1, 'cerebrovascular_disease': 1}),
    dict(label='Zone D — Elderly male, CCI=9, multiple comorbidities',
         age=82, female=0, hb=float('nan'), po4=float('nan'), cci=9, cr=838,
         cci_flags={'myocardial_infarction': 1, 'diabetes_w_complication': 1,
                    'peptic_ulcer_disease': 1, 'hemiplegia_paraplegia': 1, 'any_malignancy': 1}),
]

results = []
for p in demo_patients:
    r = predict_patient(**p)
    results.append(r)
    print(f"\n{'='*60}")
    print(f"Patient: {r['label']}")
    print(f"  Zone: {r['zone']} — {r['zone_name']}")
    print(f"  ACMM 1-year risk: {r['acmm_prob']*100:.1f}% ({r['acmm_risk_level'].upper()})")
    print(f"  RSF 1-year: A=0 → {r['rsf_R0'][0]*100:.1f}%  A=1 → {r['rsf_R1'][0]*100:.1f}%  ITE: {r['rsf_ITE'][0]*100:+.1f}pp")
    print(f"  CF 1-year ITE:  {r['cf_ite'][0]*100:+.1f}pp  CI: [{r['cf_lo'][0]*100:+.1f}, {r['cf_hi'][0]*100:+.1f}]")
    print(f"  RL 1-year ITE:  {r['rl_ite'][0]*100:+.1f}pp  CI: [{r['rl_lo'][0]*100:+.1f}, {r['rl_hi'][0]*100:+.1f}]")
    print(f"  Propensity: {r['propensity']*100:.1f}%  In overlap: {r['in_overlap']}")
    if r['caveat']:
        print(f"  Caveat: {r['caveat']}")

## 4. Counterfactual Survival Curves (RSF DR-Learner)

Figure replicating manuscript Figure 2-style per-patient curves.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
years = [1, 2, 3, 4, 5]

for ax, r in zip(axes, results):
    R0, R1 = np.array(r['rsf_R0']), np.array(r['rsf_R1'])
    ax.plot(years, (1-R0)*100, 'o-', color='#d62728', lw=2, ms=6,
            label='No early dialysis (A=0)')
    ax.plot(years, (1-R1)*100, 's-', color='#2ca02c', lw=2, ms=6,
            label='Early dialysis (A=1)')
    ax.fill_between(years, (1-R0)*100, (1-R1)*100, alpha=0.12, color='grey')
    zone_col = {'A': '#66BB6A', 'B': '#29B6F6', 'C': '#7E57C2', 'D': '#FFA726'}[r['zone']]
    ax.set_title(f"Zone {r['zone']}: {r['zone_name'][:25]}...",
                 fontsize=9, color=zone_col, fontweight='bold')
    ax.set_xlabel('Years from t₀')
    ax.set_ylim(0, 105)
    ax.set_xticks(years)
    ax.grid(alpha=0.3)
    if ax == axes[0]:
        ax.set_ylabel('Survival probability (%)')
        ax.legend(fontsize=8, loc='lower left')

plt.suptitle('Counterfactual Survival Curves — RSF DR-Learner', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../eda_output/replication_survival_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. CF vs RL Agreement

Manuscript reports Pearson r=0.979, 100% directional agreement on the spatial test set (n=1,631).

To reproduce this, load the spatial test set predictions (requires running the full evaluation pipeline).

In [ ]:
# Load spatial test predictions from the development project
# (generated by evaluate_cate_external.py in the dev project)
try:
    sp_preds = pd.read_csv('../data/spatial_test_predictions.csv')
    from scipy.stats import pearsonr
    r, p = pearsonr(sp_preds['cf_ite_1y'], sp_preds['rl_ite_1y'])
    agree = (np.sign(sp_preds['cf_ite_1y']) == np.sign(sp_preds['rl_ite_1y'])).mean()
    print(f'CF vs RL Pearson r = {r:.3f}  (p={p:.2e})')
    print(f'Directional agreement: {agree:.1%}')
    print(f'\nManuscript values: r=0.979, agreement=100%')

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(sp_preds['cf_ite_1y']*100, sp_preds['rl_ite_1y']*100,
               alpha=0.3, s=8, color='#1f77b4')
    lim = max(abs(sp_preds['cf_ite_1y'].max()*100), 60)
    ax.plot([-lim, lim], [-lim, lim], 'k--', lw=1, alpha=0.5, label='Identity')
    ax.axhline(0, color='grey', lw=0.8, alpha=0.5)
    ax.axvline(0, color='grey', lw=0.8, alpha=0.5)
    ax.set_xlabel('CF 1-year ITE (pp)')
    ax.set_ylabel('RL 1-year ITE (pp)')
    ax.set_title(f'CF vs RL Agreement (r={r:.3f})')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print('spatial_test_predictions.csv not found.')
    print('To generate: run the full evaluation pipeline in the dev project.')
    print('The file should be placed at data/spatial_test_predictions.csv')

## 6. Zone Assignment Logic

Reproducing Table 1 of the manuscript — 4-zone framework.

In [ ]:
from app.predict import ACMM_THR, ITE_LOW_THR, SUB_ITE_THR

print('PRISM 4-Zone Clinical Decision Framework (Table 1)')
print('='*60)
print(f'ACMM threshold: {ACMM_THR:.0%} (Low risk < {ACMM_THR:.0%} ≤ High risk)')
print(f'ITE threshold (low-risk patients): CF+RL avg ≤ {ITE_LOW_THR:.0%} → Zone B')
print(f'ITE threshold (high-risk patients): Subgroup ITE ≤ {SUB_ITE_THR:.4f} → Zone C')
print()

zone_table = [
    ('A', 'Low (<30%)', 'CF+RL avg', f'> {abs(ITE_LOW_THR)*100:.0f}pp', 'Conservative care'),
    ('B', 'Low (<30%)', 'CF+RL avg', f'≤ −{abs(ITE_LOW_THR)*100:.0f}pp', 'Early dialysis indicated'),
    ('C', 'High (≥30%)', 'RSF/Subgroup', f'≤ −{abs(SUB_ITE_THR)*100:.1f}pp', 'Early dialysis — strong benefit'),
    ('D', 'High (≥30%)', 'RSF/Subgroup', f'> −{abs(SUB_ITE_THR)*100:.1f}pp', 'Shared decision-making'),
]
print(f'{"Zone":<6} {"ACMM Risk":<15} {"ITE Source":<14} {"Threshold":<15} {"Recommendation"}')
print('-'*70)
for row in zone_table:
    print(f'{row[0]:<6} {row[1]:<15} {row[2]:<14} {row[3]:<15} {row[4]}')

## 7. ACMM Calibration

Reproduce the observed-to-expected (O:E) ratio from the manuscript (spatial test: O:E=0.98).

In [ ]:
from app.predict import predict_acmm
from sklearn.metrics import roc_auc_score

# Load spatial test data
# Note: data_lake/ is in the dev project; copy spatial_test_processed.csv to data/ here
sp_path = '../data/spatial_test_processed.csv'
try:
    sp = pd.read_csv(sp_path)
    FEATURES_6 = ['age_at_t0','gender','hemoglobin_at_t0','phosphate_at_t0','cci_score_total','creatinine_at_t0']

    # Generate ACMM predictions
    probs = []
    for _, row in sp.iterrows():
        p = predict_acmm(row['age_at_t0'], row['gender'], row['hemoglobin_at_t0'],
                         row['phosphate_at_t0'], row['cci_score_total'], row['creatinine_at_t0'])
        probs.append(p)
    probs = np.array(probs)

    y_1y = ((sp['event'] == 1) & (sp['duration'] <= 365)).astype(int)

    auc = roc_auc_score(y_1y, probs)
    oe  = y_1y.mean() / probs.mean()

    print(f'ACMM Spatial Test Set (n={len(sp):,})')
    print(f'  AUC-ROC:       {auc:.3f}  (manuscript: 0.793)')
    print(f'  O:E ratio:     {oe:.3f}   (manuscript: 0.98)')
    print(f'  Observed 1y:   {y_1y.mean():.3f}')
    print(f'  Expected 1y:   {probs.mean():.3f}')

    # Decile calibration plot
    df_cal = pd.DataFrame({'obs': y_1y, 'pred': probs})
    df_cal['decile'] = pd.qcut(df_cal['pred'], 10, labels=False)
    cal = df_cal.groupby('decile').agg(obs_mean=('obs','mean'), pred_mean=('pred','mean'))

    fig, ax = plt.subplots(figsize=(5,5))
    ax.scatter(cal['pred_mean']*100, cal['obs_mean']*100, s=60, color='#1f77b4', zorder=5)
    ax.plot([0,100],[0,100],'k--',lw=1,alpha=0.5,label='Perfect calibration')
    ax.set_xlabel('Predicted 1-year mortality (%)')
    ax.set_ylabel('Observed 1-year mortality (%)')
    ax.set_title(f'ACMM Calibration — Spatial Test (AUC={auc:.3f}, O:E={oe:.2f})')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print(f'{sp_path} not found.')
    print('Copy spatial_test_processed.csv from the dev project data_lake/ to data/ here.')

## 8. Sample Data Structure

Demonstrates loading the sample CSV and running predictions on all 20 synthetic patients.

In [ ]:
sample = pd.read_csv('../data/sample_patients.csv', comment='#')
print(f'Sample patients: {len(sample)}')
print(sample[['patient_id','age_years','sex','creatinine_umol_L',
              'haemoglobin_g_dL','phosphate_mmol_L','cci_total']].to_string(index=False))

In [ ]:
# Run predictions on all sample patients
CCI_FLAG_COLS = ['mi','chf','pvd','cvd','dementia','copd','pud','mild_liver',
                 'dm','dm_comp','hemiplegia','malignancy','metastatic']
CCI_FLAG_MAP = dict(zip(CCI_FLAG_COLS,
    ['myocardial_infarction','congestive_heart_failure','peripheral_vascular_disease',
     'cerebrovascular_disease','dementia','chronic_pulmonary_disease','peptic_ulcer_disease',
     'mild_liver_disease','diabetes_wo_complication','diabetes_w_complication',
     'hemiplegia_paraplegia','any_malignancy','metastatic_cancer']))

rows = []
for _, p in sample.iterrows():
    flags = {CCI_FLAG_MAP[c]: int(p[c]) for c in CCI_FLAG_COLS}
    r = predict_patient(
        age=p['age_years'], female=int(p['sex'] == 'F'),
        hb=float(p['haemoglobin_g_dL']) if not pd.isna(p['haemoglobin_g_dL']) else float('nan'),
        po4=float(p['phosphate_mmol_L']) if not pd.isna(p['phosphate_mmol_L']) else float('nan'),
        cci=p['cci_total'], cr=p['creatinine_umol_L'], cci_flags=flags,
        label=p['patient_id']
    )
    rows.append({
        'patient_id': p['patient_id'], 'zone': r['zone'],
        'acmm_risk': f"{r['acmm_prob']*100:.1f}%",
        'rsf_1y_A0': f"{r['rsf_R0'][0]*100:.1f}%",
        'rsf_1y_A1': f"{r['rsf_R1'][0]*100:.1f}%",
        'rsf_ite_1y': f"{r['rsf_ITE'][0]*100:+.1f}pp",
        'in_overlap': r['in_overlap'],
    })

results_df = pd.DataFrame(rows)
print('\nPRISM predictions for all 20 sample patients:')
print(results_df.to_string(index=False))
print('\nZone distribution:')
print(results_df['zone'].value_counts().sort_index().to_string())

## Summary

This notebook has demonstrated:
- Model loading from serialised artefacts (no raw data required)
- Single-patient inference pipeline (preprocess → predict → zone assignment)
- Replication of key metrics (AUC, O:E, CF–RL agreement)
- Batch prediction on the sample dataset

For full cohort evaluation (C-index, Brier score, AUTOC, GATES analysis), see the development project at `training/eda.py` and the scripts in `prism/scripts/`.

---

**Contact**: Ka Chun Leung — leungkc.kachun@gmail.com

**License**: MIT